# Sahformer — Colab training

Clock-aware Chessformer (faithful Maia-3 5M backbone + our time layer) on a GPU runtime.

**First:** Runtime → Change runtime type → **GPU (T4)**.

Then edit `REPO_URL` in the next cell and run top to bottom. Checkpoints stream to Google Drive.

In [ ]:
# 1) Get the code
REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"  # <-- EDIT ME
import os
repo_dir = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.isdir(repo_dir):
    !git clone "$REPO_URL"
%cd $repo_dir
!git pull --ff-only || true

In [ ]:
# 2) Deps + GPU check (torch ships with Colab)
!pip -q install python-chess zstandard
import sys; sys.path.insert(0, ".")
import torch
ok = torch.cuda.is_available()
print("torch", torch.__version__, "| cuda", ok, "|",
      torch.cuda.get_device_name(0) if ok else "NO GPU -> Runtime > Change runtime type > GPU")

## 3) Build a training shard (streamed from Lichess)

Streams a 2017-04+ month (these have `%clk` clocks) and early-stops, so only the first
chunk transfers. Notes:
- The whole raw set is held in RAM before saving (~6 KB/position, history planes dominate),
  so `MAX_POSITIONS` caps memory. ~300k positions ≈ a few GB — safe on standard Colab.
- `BALANCE=True` equalizes all 22 Elo bins to the smallest present bin; the rare extreme-Elo
  bins make that set *much* smaller. For a first real run we default `BALANCE=False`
  (more volume, Elo-skewed). Re-introduce balancing / curriculum in the eval plan.

In [ ]:
# 3) Fetch + build
URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2017-04.pgn.zst"
MAX_POSITIONS = 300000   # RAM cap (history planes dominate ~6 KB/pos)
BALANCE = False          # True = Elo-balanced but much smaller (tail-limited)

import os
os.makedirs("data", exist_ok=True)
from sahformer.download import stream_games_from_url, is_target_game
from sahformer.records import game_to_records
from sahformer.shards import records_to_arrays, balance_indices, save_shard

records = []; games = 0
for g in stream_games_from_url(URL):
    if not is_target_game(g):
        continue
    records.extend(game_to_records(g)); games += 1
    if games % 2000 == 0:
        print(f"games {games} | positions {len(records)}")
    if len(records) >= MAX_POSITIONS:
        break

arr = records_to_arrays(records)
if BALANCE:
    idx = balance_indices(arr["elo_self"], seed=0)
    arr = {k: v[idx] for k, v in arr.items()}
save_shard("data/shard.npz", arr)
print("FINAL: games", games, "| positions", arr["board"].shape[0])

## 4) Mount Drive for checkpoints

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/sahformer_ckpts"
import os; os.makedirs(OUT, exist_ok=True)
print("checkpoints ->", OUT)

## 5) Train the clock-aware model (`full`) on GPU with AMP

`best.pt` / `last.pt` stream to Drive. Bump `max_steps` for a longer run; watch the loss curve.

In [ ]:
from sahformer.training.loop import TrainConfig, train
cfg = TrainConfig(mode="full", max_steps=8000, warmup_steps=400, batch_size=256,
                  lr=3e-4, amp=True, device="cuda", out_dir=f"{OUT}/full",
                  log_every=100, ckpt_every=1000)
res = train(cfg, ["data/shard.npz"])
print("full best_total:", res["best"])

In [ ]:
import matplotlib.pyplot as plt
h = res["history"]; xs = [r["step"] for r in h]
for key in ("total", "policy", "time"):
    plt.plot(xs, [r[key] for r in h], label=key)
plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.title("full — losses"); plt.show()
plt.plot(xs, [r["move_acc"] for r in h]); plt.xlabel("step"); plt.ylabel("move_acc")
plt.title("full — move accuracy (sanity metric)"); plt.show()

## 6) Optional: baseline (clock-blind) for the later ablation comparison

In [ ]:
cfg_b = TrainConfig(mode="baseline", max_steps=8000, warmup_steps=400, batch_size=256,
                    lr=3e-4, amp=True, device="cuda", out_dir=f"{OUT}/baseline",
                    log_every=100, ckpt_every=1000)
res_b = train(cfg_b, ["data/shard.npz"])
print("baseline best:", res_b["best"], "| full best:", res["best"])

## Done

Checkpoints are on your Drive under `sahformer_ckpts/`. **Don't over-read baseline-vs-full
here** — a rigorous comparison (move-match vs Maia by Elo, think-time calibration, sampled
non-deterministic play) is the next plan. This run just trains the model for real on GPU.